In [0]:
%run ./99_Config

In [0]:
%run ./98_Utilities

In [0]:
%run ./97_Logger

In [0]:
from pyspark.sql.functions import *

pipeline_start("07_Gold_Layer")

In [0]:
silver_txn_path = f"{SILVER_PATH}/transactions"

df = read_delta(silver_txn_path)

rows_read = record_count(df)

In [0]:
daily_summary = (
    df.groupBy("transaction_date")
      .agg(
          count("*").alias("total_transactions"),
          sum("amount").alias("total_amount"),
          avg("amount").alias("average_amount")
      )
      .withColumn("gold_load_timestamp", current_timestamp())
)

branch_summary = (
    df.groupBy("branch")
      .agg(
          count("*").alias("total_transactions"),
          sum("amount").alias("total_amount")
      )
      .withColumn("gold_load_timestamp", current_timestamp())
)

customer_summary = (
    df.groupBy("customer_id")
      .agg(
          count("*").alias("transaction_count"),
          sum("amount").alias("total_amount"),
          max("amount").alias("highest_transaction")
      )
      .withColumn("gold_load_timestamp", current_timestamp())
)

In [0]:
daily_path = f"{GOLD_PATH}/daily_summary"
branch_path = f"{GOLD_PATH}/branch_summary"
customer_path = f"{GOLD_PATH}/customer_summary"

write_delta(daily_summary, daily_path, "overwrite")
write_delta(branch_summary, branch_path, "overwrite")
write_delta(customer_summary, customer_path, "overwrite")

create_delta_table("gold_daily_summary", daily_path)
create_delta_table("gold_branch_summary", branch_path)
create_delta_table("gold_customer_summary", customer_path)


In [0]:
log_success(
    notebook="07_Gold_Layer",
    dataset="transactions",
    rows_read=rows_read,
    rows_written=record_count(daily_summary),
    execution_time="Completed"
)

In [0]:
print("="*80)
print("GOLD LAYER SUMMARY")
print("="*80)
print(f"Input Rows           : {rows_read}")
print(f"Daily Summary Rows   : {record_count(daily_summary)}")
print(f"Branch Summary Rows  : {record_count(branch_summary)}")
print(f"Customer Summary Rows: {record_count(customer_summary)}")
print("="*80)

pipeline_end("07_Gold_Layer")